In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().absolute().parent))

In [ ]:
# Standard library imports
import sys
from pathlib import Path

# Third-party scientific computing
import pandas as pd

# Deep learning frameworks

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.pipeline import Pipeline

# Local imports - data processing
from src.data_models.caravanify import Caravanify, CaravanifyConfig
from src.data_models.datamodule import HydroDataModule
from src.preprocessing.grouped import GroupedTransformer
from src.preprocessing.standard_scale import StandardScaleTransformer
from src.preprocessing.log_scale import LogTransformer

# Local imports - models and evaluation
from src.models.dummy import RepeatLastValuesConfig, LitRepeatLastValues

from src.models.tft import TFTConfig, LitTFT
from src.models.ealstm import EALSTMConfig, LitEALSTM
from src.models.tide import TiDEConfig, LitTiDE
from src.models.tsmixer import TSMixerConfig, LitTSMixer
from src.model_evaluation.evaluators import TSForecastEvaluator
from src.model_evaluation.hp_from_yaml import hp_from_yaml

---

In [ ]:
STATIC_FEATURES = [
    "gauge_id",
    "p_mean",
    "area",
    "ele_mt_sav",
    "high_prec_dur",
    "frac_snow",
    "high_prec_freq",
    "slp_dg_sav",
    "cly_pc_sav",
    "aridity_ERA5_LAND",
    "aridity_FAO_PM",
]

FORCING_FEATURES = [
    "snow_depth_water_equivalent_mean",
    "surface_net_solar_radiation_mean",
    "surface_net_thermal_radiation_mean",
    "potential_evaporation_sum_ERA5_LAND",
    "potential_evaporation_sum_FAO_PENMAN_MONTEITH",
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
    "total_precipitation_sum",
]

TARGET = "streamflow"

In [ ]:
ealstm_yaml = "/Users/cooper/Desktop/CAMELS-CH/experiments/DataSharing/yaml_files/tajikistan/ealstm.yaml"
tft_yaml = "/Users/cooper/Desktop/CAMELS-CH/experiments/DataSharing/yaml_files/tajikistan/tft.yaml"
tide_yaml = "/Users/cooper/Desktop/CAMELS-CH/experiments/DataSharing/yaml_files/tajikistan/tide.yaml"
tsmixer_yaml = "/Users/cooper/Desktop/CAMELS-CH/experiments/DataSharing/yaml_files/tajikistan/tsmixer.yaml"

tft_hp = hp_from_yaml("tft", tft_yaml)
tide_hp = hp_from_yaml("tide", tide_yaml)
ealstm_hp = hp_from_yaml("ealstm", ealstm_yaml)
tsmixer_hp = hp_from_yaml("tsmixer", tsmixer_yaml)

In [ ]:
TFT_config = TFTConfig(**tft_hp)
EALSTM_config = EALSTMConfig(**ealstm_hp)
TiDE_config = TiDEConfig(**tide_hp)
TSMixer_config = TSMixerConfig(**tsmixer_hp)

dummy_config = RepeatLastValuesConfig(
    input_len=tide_hp["input_len"],
    input_size=tide_hp["input_size"],
    output_len=tide_hp["output_len"],
)

---

In [ ]:
# Either Kyrgyzstan or Tajikistan or combined
COUNTRY = "Tajikistan"

In [ ]:
config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/shapefiles",
    # human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="CA",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)

ca_caravan = Caravanify(config)
ca_basins = ca_caravan.get_all_gauge_ids()
# ca_basins = [bid for bid in ca_basins if bid not in ["CA_17110", "CA_17329"]]

print(f"Found {len(ca_basins)} total CA basins")

ca_caravan.load_stations(ca_basins)

# Prepare data frames
ts_columns = FORCING_FEATURES + [TARGET]
static_columns = STATIC_FEATURES

ca_ts_data = ca_caravan.get_time_series()[ts_columns + ["date"] + ["gauge_id"]]
ca_static_data = ca_caravan.get_static_attributes()[static_columns + ["country"]]

In [ ]:
# ids for the COUUNTRY

country_ids = ca_static_data[ca_static_data["country"] == COUNTRY]["gauge_id"].unique()

ca_ts_data = ca_ts_data[ca_ts_data["gauge_id"].isin(country_ids)]
ca_static_data = ca_static_data[ca_static_data["gauge_id"].isin(country_ids)]

print(f"Found {len(country_ids)} total CA basins in {COUNTRY}")

---

In [ ]:
# Use GroupedTransformer for both features and target
feature_pipeline = Pipeline([("scaler", StandardScaleTransformer())])

target_pipeline = GroupedTransformer(
    Pipeline([("log", LogTransformer()), ("scaler", StandardScaleTransformer())]),
    columns=[TARGET],
    group_identifier="gauge_id",
    n_jobs=-1,
)


static_pipeline = Pipeline([("scaler", StandardScaleTransformer())])
preprocessing_config = {
    "features": {"pipeline": feature_pipeline},
    "target": {"pipeline": target_pipeline},
    "static_features": {"pipeline": static_pipeline},
}

In [ ]:
STATIC_FEATURES = [col for col in static_columns]
FORCING_FEATURES = FORCING_FEATURES + [TARGET]

tft_data_module = HydroDataModule(
    time_series_df=ca_ts_data,
    static_df=ca_static_data,
    group_identifier="gauge_id",
    preprocessing_config=preprocessing_config,
    batch_size=2048,
    input_length=tft_hp["input_len"],
    output_length=tft_hp["output_len"],
    num_workers=4,
    features=FORCING_FEATURES,
    static_features=STATIC_FEATURES,
    target=TARGET,
    train_prop=0.5,
    val_prop=0.25,
    test_prop=0.25,
    max_missing_pct=10,
    min_train_years=5,
    domain_id="CA",
    use_proportional_split=True,
)

ealstm_data_module = HydroDataModule(
    time_series_df=ca_ts_data,
    static_df=ca_static_data,
    group_identifier="gauge_id",
    preprocessing_config=preprocessing_config,
    batch_size=2048,
    input_length=ealstm_hp["input_len"],
    output_length=ealstm_hp["output_len"],
    num_workers=4,
    features=FORCING_FEATURES,
    static_features=STATIC_FEATURES,
    target=TARGET,
    train_prop=0.5,
    val_prop=0.25,
    test_prop=0.25,
    max_missing_pct=10,
    min_train_years=5,
    domain_id="CA",
    use_proportional_split=True,
)

tide_data_module = HydroDataModule(
    time_series_df=ca_ts_data,
    static_df=ca_static_data,
    group_identifier="gauge_id",
    preprocessing_config=preprocessing_config,
    batch_size=2048,
    input_length=tide_hp["input_len"],
    output_length=tide_hp["output_len"],
    num_workers=4,
    features=FORCING_FEATURES,
    static_features=STATIC_FEATURES,
    target=TARGET,
    train_prop=0.5,
    val_prop=0.25,
    test_prop=0.25,
    max_missing_pct=10,
    min_train_years=5,
    domain_id="CA",
    use_proportional_split=True,
)

tsmixer_data_module = HydroDataModule(
    time_series_df=ca_ts_data,
    static_df=ca_static_data,
    group_identifier="gauge_id",
    preprocessing_config=preprocessing_config,
    batch_size=2048,
    input_length=tsmixer_hp["input_len"],
    output_length=tsmixer_hp["output_len"],
    num_workers=4,
    features=FORCING_FEATURES,
    static_features=STATIC_FEATURES,
    target=TARGET,
    train_prop=0.5,
    val_prop=0.25,
    test_prop=0.25,
    max_missing_pct=10,
    min_train_years=5,
    domain_id="CA",
    use_proportional_split=True,
)

In [ ]:
tft_regional_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/FineTuning/output/DataSharing/checkpoints/tajikistan/tft/Tajikistan_tft_epoch=10_val_loss=0.0499.ckpt"
ealstm_regional_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/FineTuning/output/DataSharing/checkpoints/tajikistan/ealstm/Tajikistan_ealstm_epoch=07_val_loss=0.0496.ckpt"
tide_regional_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/FineTuning/output/DataSharing/checkpoints/tajikistan/tide/Tajikistan_tide_epoch=12_val_loss=0.0527.ckpt"
tsmixer_regional_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/FineTuning/output/DataSharing/checkpoints/tajikistan/tsmixer/Tajikistan_tsmixer_epoch=15_val_loss=0.0494.ckpt"

tft_global_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/FineTuning/output/LowHumanInfluenceTransferDeprecated/checkpoints/tajikistan/tft/Tajikistan_tft_epoch=13_val_loss=0.0470.ckpt"
ealstm_global_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/FineTuning/output/LowHumanInfluenceTransferDeprecated/checkpoints/tajikistan/ealstm/Tajikistan_ealstm_epoch=09_val_loss=0.0494.ckpt"
tide_global_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/FineTuning/output/LowHumanInfluenceTransferDeprecated/checkpoints/tajikistan/tide/Tajikistan_tide_epoch=11_val_loss=0.0512.ckpt"
tsmixer_global_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/FineTuning/output/LowHumanInfluenceTransferDeprecated/checkpoints/tajikistan/tsmixer/Tajikistan_tsmixer_epoch=23_val_loss=0.0473.ckpt"

ealstm_benchmark_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/Benchmark/output/checkpoints/tajikistan/ealstm/run_0/Tajikistan_ealstm_epoch=33_val_loss=0.0549.ckpt"
tide_benchmark_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/Benchmark/output/checkpoints/tajikistan/tide/run_0/Tajikistan_tide_epoch=46_val_loss=0.0616.ckpt"
tsmixer_benchmark_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/Benchmark/output/checkpoints/tajikistan/tsmixer/run_0/Tajikistan_tsmixer_epoch=60_val_loss=0.0571.ckpt"
tft_benchmark_ckpt = "/Users/cooper/Desktop/CAMELS-CH/experiments/Benchmark/output/checkpoints/tajikistan/tft/run_0/Tajikistan_tft_epoch=48_val_loss=0.0527.ckpt"


In [ ]:
dummy_model = LitRepeatLastValues(config=dummy_config)
tft_regional = LitTFT.load_from_checkpoint(tft_regional_ckpt, config=TFT_config)
ealstm_regional = LitEALSTM.load_from_checkpoint(
    ealstm_regional_ckpt, config=EALSTM_config
)
tide_regional = LitTiDE.load_from_checkpoint(tide_regional_ckpt, config=TiDE_config)
tsmixer_regional = LitTSMixer.load_from_checkpoint(
    tsmixer_regional_ckpt, config=TSMixer_config
)

tft_global = LitTFT.load_from_checkpoint(tft_global_ckpt, config=TFT_config)
ealstm_global = LitEALSTM.load_from_checkpoint(ealstm_global_ckpt, config=EALSTM_config)
tide_global = LitTiDE.load_from_checkpoint(tide_global_ckpt, config=TiDE_config)
tsmixer_global = LitTSMixer.load_from_checkpoint(
    tsmixer_global_ckpt, config=TSMixer_config
)

tft_benchmark = LitTFT.load_from_checkpoint(tft_benchmark_ckpt, config=TFT_config)
ealstm_benchmark = LitEALSTM.load_from_checkpoint(
    ealstm_benchmark_ckpt, config=EALSTM_config
)
tide_benchmark = LitTiDE.load_from_checkpoint(tide_benchmark_ckpt, config=TiDE_config)
tsmixer_benchmark = LitTSMixer.load_from_checkpoint(
    tsmixer_benchmark_ckpt, config=TSMixer_config
)

# Create a dictionary mapping model names to (model, datamodule) tuples
models_and_datamodules = {
    "dummy": (dummy_model, tide_data_module),
    "tft_regional": (tft_regional, tft_data_module),
    "ealstm_regional": (ealstm_regional, ealstm_data_module),
    "tide_regional": (tide_regional, tide_data_module),
    "tsmixer_regional": (tsmixer_regional, tsmixer_data_module),
    "tft_global": (tft_global, tft_data_module),
    "ealstm_global": (ealstm_global, ealstm_data_module),
    "tide_global": (tide_global, tide_data_module),
    "tsmixer_global": (tsmixer_global, tsmixer_data_module),
    "tft_benchmark": (tft_benchmark, tft_data_module),
    "ealstm_benchmark": (ealstm_benchmark, ealstm_data_module),
    "tide_benchmark": (tide_benchmark, tide_data_module),
    "tsmixer_benchmark": (tsmixer_benchmark, tsmixer_data_module),
}


evaluator = TSForecastEvaluator(
    horizons=list(range(1, 11)),
    models_and_datamodules=models_and_datamodules,
    trainer_kwargs={"accelerator": "gpu", "devices": 1},
)

In [ ]:
# Run evaluation
results = evaluator.test_models()

In [ ]:
def filter_growing_season(eval_results):
    """
    Filter evaluation results to include only data from the growing season (April to October).

    Args:
        eval_results: Dictionary containing evaluation results with a 'df' key

    Returns:
        Dictionary with filtered dataframe and original metrics
    """
    # Create a copy of the results to avoid modifying the original
    filtered_results = eval_results.copy()

    # Extract the dataframe
    df = eval_results["df"].copy()

    # Ensure date column is datetime
    df["date"] = pd.to_datetime(df["date"])

    # Filter for growing season (April to October)
    growing_season_df = df[(df["date"].dt.month >= 4) & (df["date"].dt.month < 10)]

    # Replace the dataframe in the results
    filtered_results["df"] = growing_season_df

    return filtered_results


seasonal_tide_regional_results = filter_growing_season(results["tide_regional"])
seasonal_tide_global_results = filter_growing_season(results["tide_global"])
seasonal_tide_benchmark_results = filter_growing_season(results["tide_benchmark"])

seasonal_ealstm_regional_results = filter_growing_season(results["ealstm_regional"])
seasonal_ealstm_global_results = filter_growing_season(results["ealstm_global"])
seasonal_ealstm_benchmark_results = filter_growing_season(results["ealstm_benchmark"])

seasonal_tsmixer_regional_results = filter_growing_season(results["tsmixer_regional"])
seasonal_tsmixer_global_results = filter_growing_season(results["tsmixer_global"])
seasonal_tsmixer_benchmark_results = filter_growing_season(results["tsmixer_benchmark"])

seasonal_tft_regional_results = filter_growing_season(results["tft_regional"])
seasonal_tft_global_results = filter_growing_season(results["tft_global"])
seasonal_tft_benchmark_results = filter_growing_season(results["tft_benchmark"])

seasonal_dummy_results = filter_growing_season(results["dummy"])


seasonal_tide_regional_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_tide_regional_results["df"]
)
seasonal_tide_global_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_tide_global_results["df"]
)
seasonal_tide_benchmark_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_tide_benchmark_results["df"]
)

seasonal_ealstm_regional_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_ealstm_regional_results["df"]
)
seasonal_ealstm_global_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_ealstm_global_results["df"]
)
seasonal_ealstm_benchmark_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_ealstm_benchmark_results["df"]
)

seasonal_tsmixer_regional_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_tsmixer_regional_results["df"]
)
seasonal_tsmixer_global_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_tsmixer_global_results["df"]
)
seasonal_tsmixer_benchmark_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_tsmixer_benchmark_results["df"]
)

seasonal_tft_regional_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_tft_regional_results["df"]
)
seasonal_tft_global_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_tft_global_results["df"]
)
seasonal_tft_benchmark_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_tft_benchmark_results["df"]
)

seasonal_dummy_results["metrics"] = evaluator._calculate_overall_metrics(
    seasonal_dummy_results["df"]
)


seasonal_tide_regional_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_tide_regional_results["df"]
)
seasonal_tide_benchmark_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_tide_benchmark_results["df"]
)
seasonal_tide_global_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_tide_global_results["df"]
)

seasonal_ealstm_regional_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_ealstm_regional_results["df"]
)
seasonal_ealstm_global_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_ealstm_global_results["df"]
)
seasonal_ealstm_benchmark_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_ealstm_benchmark_results["df"]
)

seasonal_tsmixer_regional_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_tsmixer_regional_results["df"]
)
seasonal_tsmixer_global_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_tsmixer_global_results["df"]
)
seasonal_tsmixer_benchmark_results["basin_metrics"] = (
    evaluator._calculate_basin_metrics(seasonal_tsmixer_benchmark_results["df"])
)

seasonal_tft_regional_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_tft_regional_results["df"]
)
seasonal_tft_global_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_tft_global_results["df"]
)
seasonal_tft_benchmark_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_tft_benchmark_results["df"]
)

seasonal_dummy_results["basin_metrics"] = evaluator._calculate_basin_metrics(
    seasonal_dummy_results["df"]
)

seasonal_results = {}

seasonal_results["tide_regional"] = seasonal_tide_regional_results
seasonal_results["tide_global"] = seasonal_tide_global_results
seasonal_results["tide_benchmark"] = seasonal_tide_benchmark_results
seasonal_results["ealstm_regional"] = seasonal_ealstm_regional_results
seasonal_results["ealstm_global"] = seasonal_ealstm_global_results
seasonal_results["ealstm_benchmark"] = seasonal_ealstm_benchmark_results
seasonal_results["tsmixer_regional"] = seasonal_tsmixer_regional_results
seasonal_results["tsmixer_global"] = seasonal_tsmixer_global_results
seasonal_results["tsmixer_benchmark"] = seasonal_tsmixer_benchmark_results
seasonal_results["tft_regional"] = seasonal_tft_regional_results
seasonal_results["tft_global"] = seasonal_tft_global_results
seasonal_results["tft_benchmark"] = seasonal_tft_benchmark_results
seasonal_results["dummy"] = seasonal_dummy_results

In [ ]:
tide_regional_summary = evaluator.summarize_metrics(
    seasonal_results["tide_regional"]["metrics"]
)
tide_global_summary = evaluator.summarize_metrics(
    seasonal_results["tide_global"]["metrics"]
)
tide_benchmark_summary = evaluator.summarize_metrics(
    seasonal_results["tide_benchmark"]["metrics"]
)

ealstm_regional_summary = evaluator.summarize_metrics(
    seasonal_results["ealstm_regional"]["metrics"]
)
ealstm_global_summary = evaluator.summarize_metrics(
    seasonal_results["ealstm_global"]["metrics"]
)
ealstm_benchmark_summary = evaluator.summarize_metrics(
    seasonal_results["ealstm_benchmark"]["metrics"]
)

tsmixer_regional_summary = evaluator.summarize_metrics(
    seasonal_results["tsmixer_regional"]["metrics"]
)
tsmixer_global_summary = evaluator.summarize_metrics(
    seasonal_results["tsmixer_global"]["metrics"]
)
tsmixer_benchmark_summary = evaluator.summarize_metrics(
    seasonal_results["tsmixer_benchmark"]["metrics"]
)

tft_regional_summary = evaluator.summarize_metrics(
    seasonal_results["tft_regional"]["metrics"]
)
tft_global_summary = evaluator.summarize_metrics(
    seasonal_results["tft_global"]["metrics"]
)
tft_benchmark_summary = evaluator.summarize_metrics(
    seasonal_results["tft_benchmark"]["metrics"]
)
dummy_summary = evaluator.summarize_metrics(seasonal_results["dummy"]["metrics"])

---
# Plots for SDC communication strategy

In [ ]:
sns.set_context(context="paper", font_scale=1.2)

In [ ]:
def create_model_comparison_grid(
    baseline_results,
    regional_results,
    global_results,
    metric="NSE",
    models=["TiDE", "EALSTM", "TFT", "TSMixer"],
    figsize=(20, 12),
):
    """
    Create a 2x2 grid comparing the performance of four models with different data sharing strategies.
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes = axes.flatten()

    # Consistent colors across all plots
    colors = ["#BCE784", "#5DD39E", "#348AA7"]

    # Custom titles
    titles = {
        "TiDE": "TiDE Model Performance",
        "EALSTM": "EALSTM Model Performance",
        "TFT": "TFT Model Performance",
        "TSMixer": "TSMixer Model Performance",
    }

    for i, model in enumerate(models):
        # Instead of using plot_metric_summary which creates its own figure,
        # we'll directly create the plot on each subplot
        ax = axes[i]

        # Prepare data for side-by-side plotting
        data = pd.DataFrame(
            {
                "Horizon": baseline_results[model].index,
                "Baseline": baseline_results[model][metric],
                "With Kyrgyz Data": regional_results[model][metric],
                "With Global Data": global_results[model][metric],
            }
        )

        # Melt the dataframe for seaborn
        melted_data = data.melt(
            id_vars="Horizon", var_name="Dataset", value_name="Value"
        )

        # Create side-by-side bar plot on the current axis
        sns.barplot(
            x="Horizon",
            y="Value",
            hue="Dataset",
            data=melted_data,
            palette=colors,
            dodge=True,
            ax=ax,
        )

        ax.set_title(titles[model], fontsize=14, fontweight="bold")

        # Add value labels
        for j, dataset in enumerate(
            ["Baseline", "With Kyrgyz Data", "With Global Data"]
        ):
            for k, v in enumerate(data[dataset]):
                # Adjust x_offset based on the dataset
                x_offset = -0.3 if j == 0 else (0 if j == 1 else 0.3)
                ax.text(
                    k + x_offset,
                    v + 0.01,  # Small offset
                    f"{v:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )

        # Add grid for better readability
        ax.grid(axis="y", linestyle="--", alpha=0.3)

        # Enhance the axes
        ax.set_xlabel("Forecast Horizon (Days)", fontsize=12)
        ax.set_ylabel(
            f"{metric} {'(mm/d)' if metric in ['MAE', 'RMSE', 'MSE'] else ''}",
            fontsize=12,
        )

        # Make the tick labels larger and more readable
        ax.tick_params(labelsize=10)

        # Adjust y-axis limits to leave room for labels
        y_min, y_max = ax.get_ylim()
        ax.set_ylim(y_min, y_max * 1.1)

        # Remove legend from individual plots (we'll add one for the entire figure)
        ax.get_legend().remove()

    # Create a single legend for the entire figure
    handles, labels = axes[0].get_figure().gca().get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=3,
        frameon=False,
        fontsize=12,
    )

    # Adjust the spacing
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])

    return fig

In [ ]:
# Example usage:
fig = create_model_comparison_grid(
    baseline_results={
        "TiDE": tide_benchmark_summary,
        "EALSTM": ealstm_benchmark_summary,
        "TFT": tft_benchmark_summary,
        "TSMixer": tsmixer_benchmark_summary,
    },
    regional_results={
        "TiDE": tide_regional_summary,
        "EALSTM": ealstm_regional_summary,
        "TFT": tft_regional_summary,
        "TSMixer": tsmixer_regional_summary,
    },
    global_results={
        "TiDE": tide_global_summary,
        "EALSTM": ealstm_global_summary,
        "TFT": tft_global_summary,
        "TSMixer": tsmixer_global_summary,
    },
    metric="MAE",
)
sns.despine()

save_to = "/Users/cooper/Desktop/CAMELS-CH/images/SDC/model_comparison_grid.png"
plt.savefig(save_to, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def plot_metric_summary(
    summary_df1: pd.DataFrame,
    summary_df2: pd.DataFrame,
    summary_df3: pd.DataFrame,
    metric: str,
    label1: str = "Baseline",
    label2: str = "Regional Data Sharing",
    label3: str = "Global Data Sharing",
    per_basin: bool = False,
    horizons: list = None,  # New parameter: list of horizons to plot (e.g., [1, 3, 5])
    figsize=(10, 6),
    colors=None,
    title=None,
):
    """
    Plot comparison of metric summaries between three models/datasets.
    
    Args:
        summary_df1: Metric summary dataframe for baseline model
        summary_df2: Metric summary dataframe for regional data sharing
        summary_df3: Metric summary dataframe for global data sharing
        metric: Name of the metric to plot (e.g., 'NSE', 'RMSE')
        label1: Label for baseline model
        label2: Label for regional data sharing
        label3: Label for global data sharing
        per_basin: Whether to plot per-basin metrics
        horizons: List of horizon values to plot. If None, all horizons are plotted.
        figsize: Figure size as tuple (width, height)
        colors: Custom colors for the bars (list of 3 colors)
        title: Custom title for the plot
    """
    plt.figure(figsize=figsize)

    if colors is None:
        # Use a color scheme that's visually appealing and accessible
        colors = ["#BCE784", "#5DD39E", "#348AA7"]

    # Filter DataFrames if horizons is provided
    if horizons is not None:
        summary_df1 = summary_df1.loc[summary_df1.index.isin(horizons)]
        summary_df2 = summary_df2.loc[summary_df2.index.isin(horizons)]
        summary_df3 = summary_df3.loc[summary_df3.index.isin(horizons)]

    if per_basin:
        # Unstack all dataframes
        df_plot1 = summary_df1[metric].unstack(level=0)
        df_plot2 = summary_df2[metric].unstack(level=0)
        df_plot3 = summary_df3[metric].unstack(level=0)

        # Combine dataframes with a new column for dataset
        combined_df = pd.concat(
            [
                df_plot1.melt(ignore_index=False).assign(dataset=label1),
                df_plot2.melt(ignore_index=False).assign(dataset=label2),
                df_plot3.melt(ignore_index=False).assign(dataset=label3),
            ]
        )

        # Create grouped bar plot
        ax = sns.barplot(
            data=combined_df.reset_index(),
            x="horizon",
            y="value",
            hue="dataset",
            palette=colors,
            dodge=True,
        )
        if title:
            plt.title(title, fontsize=14)
        else:
            plt.title(f"{metric} by Dataset and Forecast Horizon", fontsize=14)

    else:
        # Prepare data for side-by-side plotting
        data = pd.DataFrame(
            {
                "Horizon": summary_df1.index,
                f"{label1}": summary_df1[metric],
                f"{label2}": summary_df2[metric],
                f"{label3}": summary_df3[metric],
            }
        )

        # Melt the dataframe for seaborn
        melted_data = data.melt(
            id_vars="Horizon", var_name="Dataset", value_name="Value"
        )

        # Create side-by-side bar plot
        ax = sns.barplot(
            x="Horizon",
            y="Value",
            hue="Dataset",
            data=melted_data,
            palette=colors,
            dodge=True,
        )

        if title:
            plt.title(title, fontsize=14, fontweight="bold")
        else:
            plt.title(
                f"{metric} Improvement with Data Sharing",
                fontsize=14,
                fontweight="bold",
            )

        # Add value labels
        for i, dataset in enumerate([label1, label2, label3]):
            for j, v in enumerate(data[f"{dataset}"]):
                # Adjust x_offset based on the dataset
                x_offset = -0.3 if i == 0 else (0 if i == 1 else 0.3)
                plt.text(
                    j + x_offset,
                    v + max(data[f"{dataset}"]) * 0.01,  # Small offset based on data range
                    f"{v:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )

    # Move legend below the plot
    plt.legend(
        bbox_to_anchor=(0.5, -0.15),
        loc="upper center",
        ncol=3,
        frameon=False,
        fontsize=11,
    )

    # Add grid for better readability
    plt.grid(axis="y", linestyle="--", alpha=0.3)

    # Enhance the axes
    plt.xlabel("Forecast Horizon (Days)", fontsize=12)
    plt.ylabel(
        f"{metric} {'(mm/d)' if metric in ['MAE', 'RMSE', 'MSE'] else ''}",
        fontsize=12,
    )

    # Make the tick labels larger and more readable
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)

    # Adjust y-axis limits to leave room for labels
    y_min, y_max = plt.ylim()
    plt.ylim(y_min, y_max * 1.1)

    plt.tight_layout()
    sns.despine()

    return plt


In [ ]:
plot_metric_summary(
    summary_df1=tft_benchmark_summary,
    summary_df2=tft_regional_summary,
    summary_df3=tft_global_summary,
    metric="MAE",
    label1="Baseline",
    label2="With Kyrgyz Data",
    label3="With Global Data (not including Kyrgyz)",
    per_basin=False,
    figsize=(8, 6),
    horizons=[1, 5, 10]
)

save_to = "/Users/cooper/Desktop/CAMELS-CH/images/SDC/tft_mae_comparison.png"
plt.savefig(save_to, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
ealstm_global_summary

In [ ]:
def plot_metric_summary_models(
    summary_dfs: dict,
    metric: str,
    approach_labels: dict = None,
    horizons: list = None,
    figsize=(10, 6),
    colors=None,
    title=None,
    whisker_width=0.6,
    whisker_linewidth=1.5,
    cap_size=4,
    annotate_values=True,
    annotate_delta=True,
    delta_fontsize=12,
    y_label=None,
):
    """
    Plot comparison of metric summaries across approaches with whiskers representing model spread.

    Args:
        summary_dfs: Dictionary with structure {model_name: {approach_name: dataframe}}
        metric: Name of the metric to plot (e.g., 'NSE', 'RMSE')
        approach_labels: Dictionary mapping approach names to display labels
        horizons: List of horizon values to plot. If None, all horizons are plotted.
        figsize: Figure size as tuple (width, height)
        colors: Custom colors for the bars
        title: Custom title for the plot
        whisker_width: Width of the error bars
        whisker_linewidth: Line width of the error bars
        cap_size: Size of the caps on error bars
        annotate_values: Whether to annotate the bar values
        annotate_delta: Whether to annotate delta % compared to baseline
        delta_fontsize: Font size for delta annotations
        y_label: Custom y-axis label
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import numpy as np

    # Default approach labels if not provided
    default_approach_labels = {
        "benchmark": "Baseline",
        "regional": "With Kyrgyz Data",
        "global": "With Global Data",
    }
    if approach_labels is None:
        approach_labels = default_approach_labels

    # Default colors if not provided
    if colors is None:
        colors = ["#BCE784", "#5DD39E", "#348AA7"]

    plt.figure(figsize=figsize)

    # Extract unique horizons from all dataframes
    all_horizons = set()
    for model_dict in summary_dfs.values():
        for df in model_dict.values():
            all_horizons.update(df.index.tolist())

    # Filter horizons if specified
    if horizons is not None:
        all_horizons = [h for h in all_horizons if h in horizons]

    all_horizons = sorted(all_horizons)

    # Prepare data structure for combined approach values
    approach_data = {
        approach: {h: [] for h in all_horizons} for approach in approach_labels.keys()
    }

    # Collect values for each approach across all models
    for model_name, approaches in summary_dfs.items():
        for approach_name, df in approaches.items():
            if approach_name in approach_data:
                for horizon in all_horizons:
                    if horizon in df.index:
                        approach_data[approach_name][horizon].append(
                            df.loc[horizon, metric]
                        )

    # Calculate mean and std for each approach and horizon
    combined_data = []
    for approach_name, horizon_values in approach_data.items():
        for horizon, values in horizon_values.items():
            if values:  # Only if we have values for this combination
                combined_data.append(
                    {
                        "Horizon": horizon,
                        "Approach": approach_labels.get(approach_name, approach_name),
                        "ApproachKey": approach_name,
                        "Mean": np.mean(values),
                        "Std": np.std(values) if len(values) > 1 else 0,
                        "Min": min(values),
                        "Max": max(values),
                    }
                )

    # Convert to DataFrame
    plot_df = pd.DataFrame(combined_data)

    # Create bar positions
    bar_width = 0.25  # Width of each bar
    positions = np.arange(len(all_horizons))

    # Sort approaches to ensure consistent order
    unique_approaches = list(approach_labels.values())

    # Plot grouped bars with error bars
    fig, ax = plt.subplots(figsize=figsize)

    # Track baseline means for delta calculations
    baseline_means = {}

    for i, approach in enumerate(unique_approaches):
        # Filter data for this approach
        approach_df = plot_df[plot_df["Approach"] == approach]
        
        # Store baseline values for delta calculation
        if i == 0:  # Assuming first approach is baseline
            for h, row in approach_df.iterrows():
                baseline_means[row['Horizon']] = row['Mean']

        # Calculate positions for this set of bars
        pos = positions + (i - 1) * bar_width

        means = []
        stds = []
        horizons_for_this_approach = []

        # Ensure data is in the correct order
        for horizon in all_horizons:
            horizon_data = approach_df[approach_df["Horizon"] == horizon]
            if not horizon_data.empty:
                means.append(horizon_data["Mean"].values[0])
                stds.append(horizon_data["Std"].values[0])
                horizons_for_this_approach.append(horizon)
            else:
                means.append(0)
                stds.append(0)
                horizons_for_this_approach.append(horizon)

        # Plot bars
        bars = ax.bar(
            pos,
            means,
            width=bar_width * 0.9,  # Slightly narrower for spacing
            color=colors[i],
            label=approach,
            alpha=0.8,
        )

        # Add error bars
        ax.errorbar(
            pos,
            means,
            yerr=stds,
            fmt="none",
            ecolor="#A9A9A9",
            capsize=cap_size,
            elinewidth=whisker_linewidth,
            capthick=whisker_linewidth,
            alpha=0.7,
        )
        
        # Annotate values if requested
        if annotate_values:
            for j, (m, s, h) in enumerate(zip(means, stds, horizons_for_this_approach)):
                if m > 0:  # Only annotate non-zero values
                    ax.annotate(
                        f"{m:.2f}",
                        xy=(pos[j], m + s * 1.1),
                        ha='center',
                        va='bottom',
                        fontsize=9,
                    )
        
        # Add delta % annotations for non-baseline approaches
        if annotate_delta and i > 0:  # Skip baseline
            for j, (m, h) in enumerate(zip(means, horizons_for_this_approach)):
                if h in baseline_means and baseline_means[h] > 0:
                    baseline = baseline_means[h]
                    delta = ((m - baseline) / baseline) * 100
                    # Use different colors for improvement vs degradation
                    # For MAE/RMSE/MSE, negative delta is good (reduction in error)
                    is_improvement = delta < 0 if metric in ['MAE', 'RMSE', 'MSE'] else delta > 0
                    color = "#505050"
                    
                    ax.annotate(
                        f"{delta:.1f}%",
                        xy=(pos[j], m * 0.5),  # Position in middle of bar
                        ha='center',
                        va='center',
                        fontsize=delta_fontsize,
                        color=color,
                        # fontweight='bold',
                        
                    )

    # Set x-ticks at the center of each horizon group
    ax.set_xticks(positions)
    ax.set_xticklabels(all_horizons)

    # Set title and labels
    if title:
        plt.title(title, fontsize=14, fontweight="bold")

    plt.xlabel("Forecast Horizon (Days)", fontsize=12)

    if y_label:
        plt.ylabel(y_label, fontsize=12)
    else:
        plt.ylabel(
            f"{metric} {'(mm/d)' if metric in ['MAE', 'RMSE', 'MSE'] else ''}",
            fontsize=12,
        )

    # Add grid for better readability
    plt.grid(axis="y", linestyle="--", alpha=0.3)

    # Add legend
    plt.legend(
        bbox_to_anchor=(0.5, -0.15),
        loc="upper center",
        ncol=len(unique_approaches),
        frameon=False,
        fontsize=11,
    )

    # Make the tick labels larger and more readable
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)

    # Adjust y-axis limits to leave room for labels
    y_min, y_max = plt.ylim()
    plt.ylim(y_min, y_max * 1.1)

    plt.tight_layout()
    sns.despine()

    return plt

In [ ]:
summary_dfs = {
    "tft": {
        "benchmark": tft_benchmark_summary,
        "regional": tft_regional_summary,
        "global": tft_global_summary,
    },
    "tide": {
        "benchmark": tide_benchmark_summary,
        "regional": tide_regional_summary,
        "global": tide_global_summary,
    },
    "tsmixer": {
        "benchmark": tsmixer_benchmark_summary,
        "regional": tsmixer_regional_summary,
        "global": tsmixer_global_summary,
    },
    "ealstm": {
        "benchmark": ealstm_benchmark_summary,
        "regional": ealstm_regional_summary,
        "global": ealstm_global_summary,
    },
}

plt = plot_metric_summary_models(
    summary_dfs=summary_dfs,
    metric="MAE",
    horizons=[1, 5, 10],
    approach_labels={
        "benchmark": "Baseline",
        "regional": "With Kyrgyz Data",
        "global": "With Global Data",
    },
    figsize=(10, 6),
    annotate_values=False
)

save_to = "/Users/cooper/Desktop/CAMELS-CH/images/SDC/model_comparison_mae_wiskers.png"
plt.savefig(save_to, bbox_inches='tight', dpi=300)   
plt.show()
